SINGLE AGENT SYSTEM CONTAINING MULTIPLE TOOLS


In [1]:
import os
import certifi
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from langchain.agents import create_agent




In [8]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

gemini_api_key=os.getenv("GEMINI_API_KEY")
tavily_api_key=os.getenv("Tavily_api_key")
get_weather_api_key=os.getenv("get_weather_api_key")

In [ ]:
# pre defined tool
search=TavilySearch(max_results=2)
results= search.invoke("what is the capital of india")

In [15]:
import requests
from langchain.tools import tool


In [ ]:
# custom tool to get weather data using weatherstack API

@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"http://api.weatherstack.com/current?"
        f"access_key={get_weather_api_key}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )



In [17]:
llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=gemini_api_key
)


agent=create_agent(
    model=llm,
    tools=[search,get_weather_data],
    system_prompt={
        "You are a helpful research assistant. "
        "Use Tavily web search for current or factual questions. "
        "use weather stack for current weather status of a location or city."
        "Always mention sources in the final answer."
        
    }
)

result=agent.invoke({
    "messages":[
        {
        "role":"user",
        "content":"what is the capital of india  and what is its weather right now"
        }
    ]
})


print(result["messages"][-1].content)

[{'type': 'text', 'text': 'The capital of India is New Delhi. (Source: Wikipedia via Tavily Search)\n\nThe current weather in New Delhi is Sunny with a temperature of 30°C and 35% humidity. (Source: Weatherstack)', 'extras': {'signature': 'CuABAWkUfRNuDlw+rAkDge4UnXCEjRR6U3Sx1Ya7WigIzC9wt1i/PQc8WQvnRaNsh+1ymURo9ukG8fXQYc7wfj8v0nAJNl+s0NzgsHADqFG3yICNAvAyVt4xZu9ZUWMjuDG7W1uGx7UPmUv6msrWSXAssxsXfT6MKRLMl7l2pwcIWDLEsR8YLiugF69k7+gq6GQwBNOLzK6GefyPsBUMrFp3A97hNVFrP59TlDDY5+x4cnQQHWuHTx3fhA4joEH8MeuPmItWkCFrLlULtiY08O46ZDNTL95FHwpOCYi/hoCWWmI='}}]
